# 用 Sciverse 做结构化论文筛选

> 通过 meta-catalog 获取可用字段，用 meta-search 精确过滤论文

**场景**: 用户需要按年份、期刊、作者、学科等条件精确筛选论文，类似学术搜索引擎的高级检索功能。

**使用接口**: meta-catalog, meta-search

**预估调用量**: ~2–5 次 API 调用

---


## Step 1: 环境准备

安装依赖并配置环境变量


In [ ]:
!pip install httpx
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值


## Step 2: 查询可用字段

meta-catalog 返回所有可过滤、可排序的字段及其算子


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def get_catalog():
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(f"{BASE}/meta-catalog", headers=HEADERS)
        resp.raise_for_status()
        return resp.json()

async def main():
    catalog = await get_catalog()
    print("Available fields:")
    for field in catalog["fields"]:
        print(f"  {field['name']} ({field['type']}) - operators: {field['operators']}")
    return catalog

catalog = asyncio.run(main())


## Step 3: 构造过滤条件并检索

使用 FILTER_OP_* 枚举构造 filters，SORT_ORDER_* 构造排序


In [ ]:
async def search_papers(filters: list, query: str = None, sort: list = None, page_size: int = 20):
    """调用 meta-search 进行结构化检索
    
    注意: query 和 sort 不能同时传！
    - 要相关性排序：传 query，不传 sort
    - 要字段排序（如引用数）：传 sort，不传 query
    
    filters 格式: [{field, operator, value}]
    operator 枚举: FILTER_OP_EQ / FILTER_OP_IN / FILTER_OP_GTE / FILTER_OP_LTE
    sort 格式: [{field, order}]
    order 枚举: SORT_ORDER_ASC / SORT_ORDER_DESC
    """
    async with httpx.AsyncClient(timeout=30) as client:
        body = {"filters": filters, "page_size": page_size}
        if query:
            body["query"] = query
        if sort:
            body["sort"] = sort
        resp = await client.post(
            f"{BASE}/meta-search", headers=HEADERS, json=body
        )
        resp.raise_for_status()
        return resp.json()

async def main():
    # 示例 1: 按引用数排序（不传 query）
    results = await search_papers(
        filters=[
            {"field": "publication_published_year", "operator": "FILTER_OP_GTE", "value": 2022},
            {"field": "publication_published_year", "operator": "FILTER_OP_LTE", "value": 2024},
            {"field": "publication_venue_name", "operator": "FILTER_OP_IN", "value": ["Nature", "Science"]}
        ],
        sort=[{"field": "citation_count", "order": "SORT_ORDER_DESC"}]
    )
    print(f"Found {results['total_count']} papers")
    for h in results["results"][:5]:
        print(f"  {h['title']} ({h.get('publication_published_year','')}, "
              f"{h.get('publication_venue_name','')}, "
              f"citations: {h.get('citation_count', 'N/A')})")

    # 示例 2: 按相关性排序（传 query，不传 sort）
    results2 = await search_papers(
        query="CRISPR gene editing delivery",
        filters=[
            {"field": "publication_published_year", "operator": "FILTER_OP_GTE", "value": 2023}
        ]
    )
    print(f"\
Relevance search: {results2['total_count']} papers")

asyncio.run(main())


## 注意事项

- meta-catalog 建议缓存结果（字段列表变化频率低），避免每次查询都调用
- query 和 sort 不能同时传：要相关性排序就传 query；要字段排序（如引用数）就传 sort
- filters 中的 operator 必须使用 FILTER_OP_* 枚举（如 FILTER_OP_GTE），不能用 gte/lte 等缩写
- sort 中的 order 必须使用 SORT_ORDER_ASC 或 SORT_ORDER_DESC
- 响应中论文列表字段是 results（非 hits），总数字段是 total_count（非 total）
- 常用字段名：publication_published_year、publication_venue_name、author、citation_count
- 分页使用 page 和 page_size 参数


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
